# SETUP

In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime

import os
import re

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')


# from experiments.scievo import SciEvo
from experiments.toyeoh import ToyEoH
from problems.llm4ad import (    
    CarMountainProblemSpec,
    PendulumProblemSpec,

    
    StressTrainProblemSpec,
    BactGrowProblemSpec,
    
    AdmissibleSetProblemSpec,
    VRPTWProblemSpec
)

# sim func

In [2]:
CHATGPT_4OMINI_API = dict(
            host="api.openai.com",
            key=os.environ.get('OPENAI_API_KEY'),
            model="gpt-4o-mini",
            timeout=20
        )

GEMINI_25FLASH_API = dict(
            base_url='https://generativelanguage.googleapis.com/v1beta',
            api_key=os.environ.get('GEMINI_API_KEY'),
            model='gemini-2.0-flash',
            timeout=20,
            openai_format=True
        )

In [3]:

async def nowhistles_seoh_ps(problem_spec, api, max_samples=200, hooks=[]):
    eallm = ToyEoH(
        # engine=engine,
        api=api,
        ps=problem_spec,
        hooks=hooks,
        config=dict(max_samples=max_samples)
    )
    
    best_soln, trace = await eallm.run()
    return trace


async def uniformselection_seoh_ps(problem_spec, api, max_samples=200, hooks=[]):
    eallm = ToyEoH(
        # engine=engine,
        api=api,
        ps=problem_spec,
        hooks=hooks,
        config=dict(max_samples=max_samples, unbiased_selection=True)
    )
    
    best_soln, trace = await eallm.run()
    return trace


In [4]:
import time
import datetime

class timeme:
    def __init__(self):
        self.starttime = None
    def __enter__(self):
        self.starttime = datetime.datetime.now()
    def __exit__(self, *args, **kwargs):
        print(' ==== TIMER ====')
        print(datetime.datetime.now() - self.starttime)
        
        print("===============")
        

## SciEvo2

In [5]:
async def scievo_proto(problem_spec, api, max_samples=200):
    eallm = SciEvo(
        api=api,
        ps=problem_spec,
        config=dict(
            max_samples=max_samples,
        )
    )
    
    best_soln, fancypop, trace = await eallm.run()
    return best_soln, fancypop, trace




In [6]:
from experiments.scievo2 import SciEvo

# ps = CarMountainProblemSpec(maximization=True)
ps = StressTrainProblemSpec(maximization=True)
temp = 0.75
llm_api = CHATGPT_4OMINI_API


with timeme():
    best_soln, pop, trace = await scievo_proto(ps, api=dict(temperature=temp, **llm_api), max_samples=25,)
    

HOOKUP 1
0
	filtering too short segment 1<6
HOOKUP 2
1
	filtering too short segment 1<6
HOOKUP 3
2
	filtering too short segment 1<6
HOOKUP 4
3
	filtering too short segment 1<6
HOOKUP 5
4
	filtering too short segment 1<6
HOOKUP 6
5
	filtering too short segment 3<6
	filtering too short segment 1<6
HOOKUP 7
6
	filtering too short segment 3<6
	filtering not enough seg_members 1<3
	filtering too short segment 1<6
HOOKUP 8
7
	filtering too short segment 2<6
	filtering too short segment 1<6
HOOKUP 9
8
	filtering too short segment 2<6
	filtering too short segment 2<6
	filtering too short segment 1<6
HOOKUP 10
9
	filtering too short segment 3<6
	filtering too short segment 4<6
	filtering too short segment 4<6
	filtering not enough seg_members 2<3
	filtering not enough seg_members 2<3
	filtering too short segment 1<6
HOOKUP 11
10
	filtering too short segment 2<6
	filtering too short segment 2<6
	filtering too short segment 1<6
	filtering too short segment 5<6
	filtering too short segment 5<6
	fi

<string>:15: RuntimeWarning: invalid value encountered in power
<string>:15: RuntimeWarning: invalid value encountered in power


HOOKUP 22
32
	filtering too short segment 3<6
	filtering too short segment 1<6
	filtering too short segment 3<6
	filtering too short segment 1<6
HOOKUP 23
36
	filtering too short segment 2<6
	filtering too short segment 2<6
	filtering too short segment 4<6
	filtering not enough seg_members 2<3
	filtering not enough seg_members 2<3
	filtering too short segment 1<6
HOOKUP 24
37
	filtering too short segment 3<6
	filtering too short segment 1<6
	filtering too short segment 1<6
HOOKUP 25
41
	filtering too short segment 2<6
	filtering too short segment 2<6
	filtering too short segment 1<6
	filtering too short segment 2<6
	filtering too short segment 1<6
	filtering too short segment 1<6
	filtering too short segment 5<6
	filtering too short segment 1<6
	filtering too short segment 3<6
	filtering too short segment 5<6
	filtering too short segment 3<6
	filtering too short segment 1<6
HOOKUP 26
44
	filtering too short segment 2<6
	filtering too short segment 3<6
	filtering too short segment 2<6
	

In [7]:
print(2)



2


In [8]:
%debug


> /root/eallmancy/experiments/trace_analysis/atomization.py(49)_filter_invalid()
     47         for e in self.entries:
     48             if not e.is_valid:
---> 49                 raise Exception("why are we invalidating?")
     50             #     print('invalidating',e)
     51         self.entries = [e for e in self.entries if e.is_valid]



ipdb>  self.entries


[SolutionSegment(soln_id=23, soln_length=23, start_token=7, end_token=14, original_segid=378), SolutionSegment(soln_id=27, soln_length=17, start_token=12, end_token=17, original_segid=378)]


ipdb>  exit


In [8]:
from itertools import chain

all_matches = list(set(chain(*pop.idea_extractor_cache._clustered_segments._segset_map.values())))
print(len(all_matches))



17


In [45]:
def get_seg(atm, seg):
    rr = [e for e in atm.entries if e.original_segid==seg or (e.original_segid is None and seg is None)]
    return rr.pop()

list(sorted([(a.originalseg_members(), a.solution_members(), a.token_length(), get_seg(a, None) ) for a in all_matches], key=lambda ts: ts[0]))



[({None},
  {6, 11, 26},
  4,
  SolutionSegment(soln_id=26, soln_length=30, start_token=11, end_token=16, original_segid=None)),
 ({369, 373, 377, None},
  {6, 9, 11, 37},
  6,
  SolutionSegment(soln_id=37, soln_length=28, start_token=15, end_token=20, original_segid=None)),
 ({331, 333, None},
  {16, 18, 32},
  10,
  SolutionSegment(soln_id=32, soln_length=28, start_token=15, end_token=24, original_segid=None)),
 ({382, None},
  {10, 17, 41},
  6,
  SolutionSegment(soln_id=17, soln_length=24, start_token=11, end_token=15, original_segid=None)),
 ({382, None},
  {10, 17, 41},
  7,
  SolutionSegment(soln_id=17, soln_length=24, start_token=12, end_token=17, original_segid=None)),
 ({601, 609, None},
  {10, 17, 51},
  7,
  SolutionSegment(soln_id=51, soln_length=26, start_token=16, end_token=22, original_segid=None)),
 ({567, 579, None},
  {9, 26, 47},
  7,
  SolutionSegment(soln_id=47, soln_length=36, start_token=19, end_token=25, original_segid=None)),
 ({89, None},
  {6, 9, 11},
  6,
 

In [10]:
import random
from experiments.trace_analysis.segm_printing import *


def _print(atm, ttrace, subm_stats):

    str_lines = []
    for xxx, s_entry in enumerate(atm.entries):
        if not xxx==0:
            str_lines.append('------------------------------'*2)

        prelude = f"Solution {s_entry.soln_id}"
        text = (simple_segm_substr(
            ttrace[s_entry.soln_id].program,
            s_entry.start_token, s_entry.end_token,
            subm_stats['submissions'][s_entry.soln_id]['tokenList']
        ))
        str_lines.append(prelude)
        str_lines.append(text)
    str_lines.append('------------------------------'.replace('-','='))
    str_lines.append(atm.description)
    return ('\n'.join(str_lines))

In [41]:
seln = random.choice(all_matches)
print(seln.solution_members())
print(seln.originalseg_members())
print([e.start_token for e in seln.entries])
print()
print(_print(seln, trace, pop.idea_extractor_cache._tcmp))



{41, 10, 17}
{None, 382}
[12, 12, 12]

Solution 41
    elastic_contrib = params[0] * strain + params[1] * (strain ** 2) + params[2] * temp + params[3] * (temp ** 2)
    plastic_contrib = params[4] * (strain - transition_point) + params[6]
    
    transition_factor = 1 / (1 + np.exp(-params[7] * (strain - transition_point)))
------------------------------------------------------------
Solution 10
    elastic_contrib = params[0] * strain + params[1] * temp + params[2] * (strain ** 2) + params[3] * (temp ** 2)
    plastic_contrib = params[4] * strain + params[6] * temp
    
    transition_factor = 1 / (1 + np.exp(-params[7] * (strain - transition_point)))
------------------------------------------------------------
Solution 17
    elastic_contrib = params[0] * strain + params[1] * temp + params[2] * (strain ** 2) + params[3] * (strain * temp) + params[4] * (temp ** 2)
    plastic_contrib = params[6] * strain + params[7] * temp
    
    transition_factor = 1 / (1 + np.exp(-params[8] * (st

In [11]:
[e.length for e in seln.entries]


[10, 10, 10, 10]

In [18]:
len(' * temp`')


8

In [ ]:
print_atom(all_matches[3], trace[2], trace[1].idea_extractor_cache._tcmp)

In [ ]:


def prmpt_print_atom(atm, ttrace, subm_stats):

    str_lines = []
    for xxx, s_entry in enumerate(atm.entries):
        if not xxx==0:
            str_lines.append('------------------------------'*2)

        prelude = f"Solution {s_entry.soln_id}"
        text = (simple_segm_substr(
            ttrace.list[s_entry.soln_id].program,
            s_entry.start_token, s_entry.end_token,
            subm_stats['submissions'][s_entry.soln_id]['tokenList']
        ))
        str_lines.append(prelude)
        str_lines.append(text)
    return ('\n'.join(str_lines))



def analyze_atom_p1(atm, ttrace, tcmp):
    resp = get_openai_chat_completion(f"""
A team of scientists have been working on implementations of the following function:
```\n{test_template}```

After some analysis, we have noticed some similar code segments across several solutions:
{prmpt_print_atom(atm, ttrace, tcmp)}

Analyzing the code segments above, provide a list of clustered similar code segments and a summary of their unifying ideas.
This list should be in markdown format starting with solution indices in a code block, like in the example below (with lorem ipsum taking the spot of the verbal analysis)
```
- [22,37,45] lorem ipsum dolor sit amit
```
""")
    return resp

In [ ]:
build_artifact(self, prompt: str, arte_class, parents: List[Artifact] = [], register=True)

In [13]:
trace


(<experiments.scievo2.base.ThoughtfulSolution at 0x7925ff3718b0>,
 <eallmancy.engine.engine_trace.EngineTrace at 0x79260c3957f0>)

In [20]:
print_atom(all_matches[3], trace[2], trace[1].idea_extractor_cache._tcmp)

import numpy as np

def equation(strain: np.ndarray, temp: np.ndarray, params: np.ndarray) -> np.ndarray:
    """ Mathematical function for stress in Aluminium rod
    Args:
        strain: A numpy array representing observations of strain.
        temp: A numpy array representing observations of temperature.
        params: Array of numeric constants or parameters to be optimized

    Return:
        A numpy array representing stress as the result of applying the mathematical function to the inputs.
    """
    return params[0] * strain  +  params[1] * temp
------------------------------
import numpy as np

def equation(strain: np.ndarray, temp: np.ndarray, params: np.ndarray) -> np.ndarray:
    """ Mathematical function for stress in Aluminium rod
    Args:
        strain: A numpy array representing observations of strain.
        temp: A numpy array representing observations of temperature.
        params: Array of numeric constants or parameters to be optimized

    Return:
       

In [5]:
%run -i /root/eallmancy/experiments/trace_analysis/idea_extractor.py

In [6]:

# ps = CarMountainProblemSpec(maximization=True)
ps = StressTrainProblemSpec(maximization=True)
temp = 0.75
llm_api = CHATGPT_4OMINI_API

idea_extractor_cache = IdeaExtractorCache(ps, ignore_jplagapi_status=True)

with timeme():
    trace = await uniformselection_seoh_ps(ps, api=dict(temperature=temp, **llm_api), max_samples=15, hooks=[idea_extractor_cache.hook])
    

HOOKUP 1
0
	filtering not enough seg_members 1<4
HOOKUP 2
1
	filtering not enough seg_members 2<4
	filtering not enough seg_members 2<4
HOOKUP 3
2
	filtering not enough seg_members 3<4
	filtering not enough seg_members 3<4
HOOKUP 4
3
HOOKUP 5
4
HOOKUP 6
5
HOOKUP 7
6
	filtering not enough seg_members 1<4
HOOKUP 8
7
	filtering too short segment 1<7
HOOKUP 9
8
	filtering too short segment 1<7
HOOKUP 10
9
	filtering not enough seg_members 2<4
	filtering not enough seg_members 2<4
HOOKUP 11
10
	filtering not enough seg_members 3<4
	filtering not enough seg_members 3<4
	filtering not enough seg_members 3<4
	filtering not enough seg_members 1<4
HOOKUP 12
11
	filtering too short segment 4<7
HOOKUP 13
12
	filtering not enough seg_members 1<4
HOOKUP 14
13
	filtering too short segment 3<7
HOOKUP 15
14
	filtering not enough seg_members 2<4
HOOKUP 16
15
	filtering too short segment 1<7
 ==== TIMER ====
0:01:27.294848


In [7]:

# ps = CarMountainProblemSpec(maximization=True)
ps = StressTrainProblemSpec(maximization=True)

temp = 0.75
llm_api = CHATGPT_4OMINI_API

idea_extractor_cache = IdeaExtractorCache(ps, ignore_jplagapi_status=True)

with timeme():
    # without trace analysis timing
    _
    trace = await uniformselection_seoh_ps(ps, api=dict(temperature=temp, **llm_api), max_samples=15, hooks=[])
    

 ==== TIMER ====
0:01:25.387600


In [31]:

idxs = []
for a in idea_extractor_cache._clustered_segments._segset_map.keys():
    for b in range(len(idea_extractor_cache._clustered_segments._segset_map[a])):
        idxs.append((a,b))
        #idea_extractor_cache._clustered_segments._segset_map[a][b]
        

In [32]:
import random

print(len(idea_extractor_cache._clustered_segments._segset_map.values()))

2


In [35]:
cci = random.choice(idxs)

test_atom = idea_extractor_cache._clustered_segments._segset_map[cci[0]][cci[1]]
print(set(e.original_segid for e in test_atom.entries))
print(prmpt_print_atom(test_atom, trace, idea_extractor_cache._tcmp))
print()
print(analyze_atom_p1(test_atom, trace, idea_extractor_cache._tcmp, idea_extractor_cache._template_program))

{None, 81, 82, 83}
Solution 0
import numpy as np

def choose_action(pos: float, v: float, last_action: int) -> int:
    """Return the action for the car to proceed the next move.
    Args:
        pos: Car's position, a float ranges between [-1.2, 0.6].
        v: Car's velocity, a float ranges between [-0.07, 0.07].
        last_action: Car's next move, a int ranges between [0, 1, 2].
    Return:
         An integer representing the selected action for the car.
         0: accelerate to left
         1: don't accelerate
         2: accelerate to right
    """
    if pos < -1.2:
        return 2  # Accelerate to the right if too far left
    elif pos > 0.6:
        return 0  # Accelerate to the left if too far right
    elif v < 0:
        return 2 if last_action != 0 else 1  # Change direction if moving left
    else:
        return 0 if last_action != 2 else 1  # Change direction if moving right
------------------------------------------------------------
Solution 2
import numpy as n

In [23]:
test_atom.entries[0]

SolutionSegment(soln_id=18, soln_length=23, start_token=0, end_token=6, original_segid=None)

In [20]:
test_atom.solution_members()

[18, 26, 12, 29]

'\nimport numpy as np\n\ndef choose_action(pos: float, v: float, last_action: int) -> int:\n    """Return the action for the car to proceed the next move.\n    Args:\n        pos: Car\'s position, a float ranges between [-1.2, 0.6].\n        v: Car\'s velocity, a float ranges between [-0.07, 0.07].\n        last_action: Car\'s next move, a int ranges between [0, 1, 2].\n    Return:\n         An integer representing the selected action for the car.\n         0: accelerate to left\n         1: don\'t accelerate\n         2: accelerate to right\n    """\n    return np.random.randint(3)\n'

In [7]:
dir(idea_extractor_cache)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_all_template_comparisons',
 '_clustered_segments',
 '_scrape_idea_segments',
 '_soln_atoms',
 '_tcmp',
 '_template_program',
 '_update_atoms',
 '_update_jplag',
 'hook',
 'ignore_jplagapi_status']

In [7]:
import jplag

In [10]:
jplag.api_handler.is_alive

False

In [18]:
trace.list

# sim running

In [ ]:
llms_to_run = {
    '4omini': CHATGPT_4OMINI_API,
    # 'gemini25f': GEMINI_25FLASH_API
}

ps_to_test = [

    CarMountainProblemSpec(maximization=True),
    # PendulumProblemSpec(maximization=True),
    
    # StressTrainProblemSpec(maximization=True),
    # BactGrowProblemSpec(maximization=True),
    
    ## AdmissibleSetProblemSpec(maximization=True, dimension=21, weight=15),
    ## AdmissibleSetProblemSpec(maximization=True, dimension=12, weight=7),
    # AdmissibleSetProblemSpec(maximization=True, dimension=15, weight=10),
    # VRPTWProblemSpec(maximization=True)
]


res = defaultdict(lambda : [])

for llm_name, llm_api in llms_to_run.items():
    for idx, ps in enumerate(ps_to_test):
        for temp in [0.0, 0.5, 1.0]:
            for _ in range(3):
            
                trace = await uniformselection_seoh_ps(ps, api=dict(temperature=temp, **llm_api), max_samples=200)
                res[ps.__class__.__name__].append(trace)
                time = datetime.datetime.now().strftime('%Y%m%d-%H%S')
                print(f'[{time}]', llm_name, ps.__class__.__name__, max([a.fitness for a in trace.list]))
                
                trace.save(f'/root/eallmancy/experiments/toyeoh/sims/temperatures2/{llm_name}{int(100*temp)}_{ps.__class__.__name__}_UNIFORM-{time}.pk')
    

/root/reallm/venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [ ]:
# sim started at 9:31

In [1]:
!ls /root/eallmancy/experiments/toyeoh/sims/temperatures

4omini50_PendulumProblemSpec_UNIFORM-20250710-0057.pk
4omini50_PendulumProblemSpec_UNIFORM-20250710-0354.pk
4omini50_PendulumProblemSpec_UNIFORM-20250710-0512.pk
4omini50_StressTrainProblemSpec_UNIFORM-20250710-0708.pk
4omini60_PendulumProblemSpec_UNIFORM-20250710-0137.pk
4omini60_PendulumProblemSpec_UNIFORM-20250710-0338.pk
4omini60_PendulumProblemSpec_UNIFORM-20250710-0537.pk
4omini60_StressTrainProblemSpec_UNIFORM-20250710-0707.pk
4omini70_PendulumProblemSpec_UNIFORM-20250710-0109.pk
4omini70_PendulumProblemSpec_UNIFORM-20250710-0423.pk
4omini70_PendulumProblemSpec_UNIFORM-20250710-0648.pk
4omini70_StressTrainProblemSpec_UNIFORM-20250710-0730.pk
4omini80_PendulumProblemSpec_UNIFORM-20250710-0224.pk
4omini80_PendulumProblemSpec_UNIFORM-20250710-0414.pk
4omini80_PendulumProblemSpec_UNIFORM-20250710-0655.pk
4omini90_PendulumProblemSpec_UNIFORM-20250710-0213.pk
4omini90_PendulumProblemSpec_UNIFORM-20250710-0537.pk
4omini90_PendulumProblemSpec_UNIFORM-20250710-0703.pk


In [5]:
import datetime

print(datetime.datetime.now())

2025-06-28 19:38:45.328147


### scratch